### Setup and Data Loading

In [1]:
# Standard libraries
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn import set_config
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder

# Config
set_config(transform_output='pandas')

# Reading Data
url = 'https://drive.google.com/file/d/1p6zD4i4wPRfxiybxSUrmZTag2R4bLy90/view?usp=sharing'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
data = pd.read_csv(path)

# Defining X and y
X = data
y = X.pop('Expensive')

# Data splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3146)


### Inspecting Data

In [2]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1168 entries, 179 to 507
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   LotArea       1168 non-null   int64  
 1   LotFrontage   967 non-null    float64
 2   TotalBsmtSF   1168 non-null   int64  
 3   BedroomAbvGr  1168 non-null   int64  
 4   Fireplaces    1168 non-null   int64  
 5   PoolArea      1168 non-null   int64  
 6   GarageCars    1168 non-null   int64  
 7   WoodDeckSF    1168 non-null   int64  
 8   ScreenPorch   1168 non-null   int64  
 9   MSZoning      1168 non-null   object 
 10  Condition1    1168 non-null   object 
 11  Heating       1168 non-null   object 
 12  Street        1168 non-null   object 
 13  CentralAir    1168 non-null   object 
 14  Foundation    1168 non-null   object 
dtypes: float64(1), int64(8), object(6)
memory usage: 146.0+ KB


Objects / Strings:
* Condition1
* Heating
* Street
* CentralAir
* Foundation

### Automating Pipelines

Defining the branches

In [3]:
# 1. Select categorical and numerial column NAMES
X_cat_columns = X.select_dtypes(exclude='number').columns
X_num_columns = X.select_dtypes(include='number').columns

# 2. Create numerical pipeline
numeric_pipe = make_pipeline(
    SimpleImputer(strategy='mean')
)

# 3. Create categorical pipeline
categoric_pipe = make_pipeline(
    SimpleImputer(strategy='constant',
                  fill_value='N_A'),
    OneHotEncoder(sparse_output=False,
                  handle_unknown='ignore')
)

Creating the Preprocessor with `make_column_transformer`

In [4]:
preprocessor = make_column_transformer(
    (numeric_pipe, X_num_columns),
    (categoric_pipe, X_cat_columns)
)
preprocessor

ColumnTransformer(transformers=[('pipeline-1',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer())]),
                                 Index(['LotArea', 'LotFrontage', 'TotalBsmtSF', 'BedroomAbvGr', 'Fireplaces',
       'PoolArea', 'GarageCars', 'WoodDeckSF', 'ScreenPorch'],
      dtype='object')),
                                ('pipeline-2',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='N_A',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['MSZoning', 'Condition1', 'Heating', 'Street', 'CentralAir',
       'Foundation'],
      dtype='object'))])

Building the Full Pipeline

In [5]:
full_pipeline = make_pipeline(preprocessor,
                              DecisionTreeClassifier(random_state=42))

full_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer())]),
                                                  Index(['LotArea', 'LotFrontage', 'TotalBsmtSF', 'BedroomAbvGr', 'Fireplaces',
       'PoolArea', 'GarageCars', 'WoodDeckSF', 'ScreenPorch'],
      dtype='object')),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='N_A',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['MSZoning', 'Condition1', 'Heating', 'Street', 'CentralAir',
       'Foundation'],
      dtype='object'))])),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(random_state=42))])

Fit the X_train & y_train data with the pipeline

In [6]:
full_pipeline.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer())]),
                                                  Index(['LotArea', 'LotFrontage', 'TotalBsmtSF', 'BedroomAbvGr', 'Fireplaces',
       'PoolArea', 'GarageCars', 'WoodDeckSF', 'ScreenPorch'],
      dtype='object')),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='N_A',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['MSZoning', 'Condition1', 'Heating', 'Street', 'CentralAir',
       'Foundation'],
      dtype='object'))])),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(random_state=42))])

### Using New Pipeline with `GridSearchCV`

In [7]:
# 1. Define parameter grid
param_grid = {
    'columntransformer__pipeline-1__simpleimputer__strategy': ['mean', 'median'],
    'decisiontreeclassifier__max_depth': range(2, 14, 2),
    'decisiontreeclassifier__min_samples_leaf': range(3, 20, 3)
}

# 2. Define GridSearchCV
search = GridSearchCV(full_pipeline,
                      param_grid,
                      cv=5,
                      verbose=1)

search.fit(X_train, y_train)

print(f'Best score: {search.best_score_:.4f}')
print('Best parameters found:')
print(search.best_params_)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best score: 0.9212
Best parameters found:
{'columntransformer__pipeline-1__simpleimputer__strategy': 'mean', 'decisiontreeclassifier__max_depth': 4, 'decisiontreeclassifier__min_samples_leaf': 9}


best cross-validation score:
* `max_depth`of 4
* `min_samples_leaf` of 9
* `mean` strategy


### Accuracy TEST set

In [8]:
y_pred = search.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.4f}')

Test accuracy: 0.9041


### BONUS: Inspecting Pipeline Steps

In [9]:
# Access the 'columntransformer' step
full_pipeline.named_steps.columntransformer

ColumnTransformer(transformers=[('pipeline-1',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer())]),
                                 Index(['LotArea', 'LotFrontage', 'TotalBsmtSF', 'BedroomAbvGr', 'Fireplaces',
       'PoolArea', 'GarageCars', 'WoodDeckSF', 'ScreenPorch'],
      dtype='object')),
                                ('pipeline-2',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='N_A',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['MSZoning', 'Condition1', 'Heating', 'Street', 'CentralAir',
       'Foundation'],
      dtype='object'))])

Going deeper! Beware of lower and upper case.

In [10]:
(
    full_pipeline
    .named_steps.columntransformer
    .named_transformers_['pipeline-2']
    .named_steps.onehotencoder
    .get_feature_names_out()
)

array(['MSZoning_C (all)', 'MSZoning_FV', 'MSZoning_RH', 'MSZoning_RL',
       'MSZoning_RM', 'Condition1_Artery', 'Condition1_Feedr',
       'Condition1_Norm', 'Condition1_PosA', 'Condition1_PosN',
       'Condition1_RRAe', 'Condition1_RRAn', 'Condition1_RRNe',
       'Condition1_RRNn', 'Heating_Floor', 'Heating_GasA', 'Heating_GasW',
       'Heating_Grav', 'Heating_OthW', 'Heating_Wall', 'Street_Grvl',
       'Street_Pave', 'CentralAir_N', 'CentralAir_Y', 'Foundation_BrkTil',
       'Foundation_CBlock', 'Foundation_PConc', 'Foundation_Slab',
       'Foundation_Stone', 'Foundation_Wood'], dtype=object)